In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from datasets.arrow_dataset import Dataset as HFDataset

from torchvision.transforms import v2

from torch import Tensor
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from vit import *
from train import *
from eval import *

from IPython.display import clear_output

import sys


train_ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="train",
)

val_ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="validation",
)

test_ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="test",
)

num_train_samples = len(train_ds)
num_val_samples = len(val_ds)
num_test_samples = len(test_ds)


In [ ]:
class ImageNetDataset(Dataset):
    def __init__(self, dataset: HFDataset, transform: v2.Transform = None) -> None:
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx: int) -> tuple[Tensor, int]:
        sample = self.dataset[idx]

        image = sample["image"]
        label = sample["label"]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

eval_transform = v2.Compose([
    v2.ToImage(),

    # preserves aspect ratio, addresses varying image resolutions
    v2.Resize(size=256, interpolation=v2.InterpolationMode.BICUBIC),
    v2.CenterCrop(size=224),

    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

train_transform = v2.Compose([
    v2.ToImage(),

    v2.RandomResizedCrop( # defaults
        size=224,
        scale=(0.08, 1.0),
        ratio=(3/4, 4/3),
        interpolation=v2.InterpolationMode.BICUBIC
    ),
    v2.RandomHorizontalFlip(p=0.5),

    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

mixup = v2.MixUp(alpha=1.0, num_classes=1000)

clean_train_dataset = ImageNetDataset(train_ds, eval_transform)
train_dataset = ImageNetDataset(train_ds, train_transform)

val_dataset = ImageNetDataset(val_ds, eval_transform)
test_dataset = ImageNetDataset(test_ds, eval_transform)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers=16,
    pin_memory=True
)

clean_train_dataloader = DataLoader(
    clean_train_dataset,
    batch_size = batch_size,
    shuffle = False,
    num_workers=16,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle = False,
    num_workers=16,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = False,
    num_workers=16,
    pin_memory=True
)


device = "cuda" if torch.cuda.is_available() else "cpu"

torch.set_float32_matmul_precision("high")

config = ViTConfig(
    n_heads = 4,
    embd_size = 128,
    patch_size = 16,
    dropout = 0.2,
    num_encoders = 4,
    input_shape = (224, 224, 3),
    output_size = 1000,

)
model = ViT(config).to(device)
model = torch.compile(model) if device == "cuda" else model

epochs = 1000
lr = 1e-3
weight_decay = 1e-2


optimizer = torch.optim.AdamW(
    params=model.parameters(), 
    lr=lr, 
    weight_decay=weight_decay
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0= 500,
    T_mult = 1,
    eta_min = 1e-5
)
loss_fn = torch.nn.CrossEntropyLoss()



plot_metrics = {
    "train_losses": [],
    "val_losses": [],

    "train_accs": [],
    "val_accs": [],

    "train_epochs": [],
    "val_epochs": [],
}


for epoch in range(epochs):
    current_lr = scheduler.get_last_lr()[0]
    pbar = tqdm(total=num_train_samples, unit="samples", desc = f"Epoch: {epoch+1}")

    train_model(
        model,
        train_dataloader,
        optimizer, loss_fn, device,
        pbar, plot_metrics,
        scheduler=scheduler,
        mixup=mixup
    )
    

    if (epoch == 0 or (epoch+1) % 5 == 0):
        evaluate_train_and_val(
            model, 
            clean_train_dataloader, val_dataloader,
            num_train_samples, num_val_samples,
            loss_fn, device, epoch,
            plot_metrics
        )

        clear_output(wait=True)

        print(f"Epoch: {epoch+1} | "
              f"train_loss: {plot_metrics["train_losses"][-1] :.4f} | "
              f"train_acc: {plot_metrics["train_accs"][-1] :.2f} | "
              f"val_loss: {plot_metrics["val_losses"][-1] :.4f} | "
              f"val_acc: {plot_metrics["val_accs"][-1] :.2f} | "
              f"lr: {current_lr :.6f}\n"
        ) 


        plt.figure(figsize = (12, 6))

        plt.subplot(121)
        plt.plot(plot_metrics["train_epochs"], plot_metrics["train_losses"], label = "train loss")
        plt.plot(plot_metrics["val_epochs"], plot_metrics["val_losses"], label = "val loss")
        plt.xlabel("Epoch")
        plt.legend()

        plt.subplot(122)
        plt.plot(plot_metrics["val_epochs"], plot_metrics["train_accs"], label = "train acc")
        plt.plot(plot_metrics["val_epochs"], plot_metrics["val_accs"], label = "val acc")
        plt.xlabel("Epoch")
        plt.legend()

        plt.show()